# 08 – Robustheitschecks

**Robustheitschecks auf Wunsch des Betreuers.** Sie werten den Testzeitraum erneut aus und dienen **nicht** der Modellauswahl: Die Hauptmodelle A, B, B2 und C bleiben, wie sie sind, egal wie die Checks ausfallen.

Für jeden Check:
- **Koeffizienten (Training):** Spezifikation wie **B2** (Modell B plus zentriertes `log_assets²`), Standardfehler nach Firma geclustert.
- **Test (einmal):** ROC-AUC und PR-AUC von **Modell B** (ohne Größe²) in derselben Variante, mit 95 %-Intervall per Firmen-Bootstrap. B2 selbst bleibt – wie festgelegt – ohne Test-Auswertung; so sind die Testwerte direkt mit Modell B aus Notebook 06 (ROC-AUC 0,698) vergleichbar.
- **Vorverarbeitung wie in 04/06**, aber für jede Variante **neu aus ihrem eigenen Training** bestimmt: Kappungsgrenzen (1./99. Perzentil), Mediane, Indikator-Schwelle (>2 % fehlend und ≥ 5 Positive in der Lücke), Wegfall neuer Indikatoren mit |rho| > 0,7 zu einem Palepu-Indikator, Zentrierungs-Mittelwert für die Größe. Auch die Jahresmediane für `grdummy` werden in der jeweiligen Stichprobe neu berechnet.
- `idummy` bleibt in allen Checks gleich definiert (pe_main-Events im Vorjahr in derselben 2-stelligen SIC): Er ist eine erklärende Variable, kein Outcome.

**Zeiträume bei anderen Horizonten.** Ein Label mit Horizont h schaut h Jahre voraus. Damit sich die Ereignisse von Training und Test nicht überlappen, muss das letzte Trainingsjahr + h vor dem ersten Testjahr + 1 liegen.
- **h = 1:** Training FY2012–2017 (Ereignisse bis 2018), Test FY2020–2022 (Ereignisse ab 2021) – dieselben Jahre wie im Hauptmodell. Ein Embargo wäre hier nicht nötig; es bleibt, damit Training und Test genau dieselben Jahre wie im Hauptmodell haben.
- **h = 5:** Labels sind nur bis FY2020 vollständig beobachtet (2020 + 5 = 2025). Test daher nur **FY2020** (Ereignisse 2021–2025); das Training muss vor 2021 enden, also **FY2012–2015** (Ereignisse bis 2020), Embargo FY2016–2019.

## A. Daten laden und Variablen bauen (Logik wie Notebooks 03, 04 und 06)

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score

DATA = Path("..") / "data"
panel = pd.read_csv(DATA / "processed" / "firm_year_panel.csv", dtype={"cik": str})
B_NEW = ["fcf_to_assets", "fcf_vol", "interest_coverage", "ppe_to_assets", "rd_to_assets", "cash_ratio", "altman_z2"]
ratios = pd.read_csv(DATA / "processed" / "firm_year_ratios.csv",
                     usecols=["accession", "sales_growth_1y", "leverage", "log_assets", "float_to_book", "float_to_earnings",
                              "no_interest_expense"] + B_NEW)            # no_interest_expense nur für Check 9
fund = pd.read_csv(DATA / "processed" / "firm_year_fundamentals.csv",
                   usecols=["accession", "net_income", "equity", "cash", "liabilities_current", "total_assets"])
full = (panel.merge(ratios, on="accession", validate="one_to_one")
             .merge(fund, on="accession", validate="one_to_one"))
full["roe"] = full["net_income"] / full["equity"].where(full["equity"] > 0)
full["liquidity"] = (full["cash"] - full["liabilities_current"]) / full["total_assets"].where(full["total_assets"] > 0)

# idummy (Notebook 03): pe_main-Event (first_event_date) in derselben 2-stelligen SIC im Kalenderjahr davor
events = pd.read_csv(DATA / "interim" / "combined_events.csv", dtype={"cik": str})
events = events[events["sponsor_like_ptp"].astype(str) == "True"].copy()
events["event_date"] = pd.to_datetime(events["first_event_date"])
events["event_year"] = events["event_date"].dt.year
filings = pd.read_csv(DATA / "interim" / "stage_11_raw_quarters.csv", dtype={"cik": str},
                      usecols=["cik", "sic", "filed_date"]).dropna(subset=["sic"])
filings["filed_date"] = pd.to_datetime(filings["filed_date"])
f = filings.merge(events[["cik", "event_date"]], on="cik")
f["before"] = f["filed_date"] <= f["event_date"]
f["gap"] = (f["filed_date"] - f["event_date"]).abs()
f = pd.concat([f[f["before"]].sort_values("filed_date", ascending=False).groupby("cik").head(1),
               f[~f["before"]].sort_values("gap").groupby("cik").head(1)]).drop_duplicates("cik")
events = events.merge(f[["cik", "sic"]], on="cik", how="left")
active = (events.dropna(subset=["sic"]).assign(sic2=lambda e: e["sic"] // 100)
                .groupby(["sic2", "event_year"]).size().rename("n_events").reset_index())
full["sic2"] = pd.to_numeric(full["sic"], errors="coerce") // 100
full["lookup_year"] = full["fiscal_year"] - 1
full = full.merge(active.rename(columns={"event_year": "lookup_year"}), on=["sic2", "lookup_year"], how="left")
full["idummy"] = (full["n_events"].fillna(0) > 0).astype(float).where(full["sic2"].notna())
# Partnership-Kennzeichen über den Firmennamen (Entscheidung 30.10.2026), Muster:
#   LP / L.P. / L P als eigenes Wort (danach Ende, Leerzeichen, Komma, Klammer oder "/"),
#   LIMITED PARTNERSHIP, PARTNERSHIP am Namensende, MLP als eigenes Wort. Groß-/Kleinschreibung egal.
import re
PARTNERSHIP_PATTERN = re.compile(r"\bL\.?\s?P\.?(?=$|[\s,)/])|LIMITED PARTNERSHIP|\bPARTNERSHIP\W*$|\bMLP\b", re.I)
full["is_partnership"] = full["company_name"].str.contains(PARTNERSHIP_PATTERN)
print(f"Panel mit allen Rohwerten: {len(full):,} Zeilen; Partnership-Zeilen: {full['is_partnership'].sum():,}")

Panel mit allen Rohwerten: 71,713 Zeilen; Partnership-Zeilen: 1,375


In [2]:
PALEPU = ["roe", "sales_growth_1y", "liquidity", "leverage", "grdummy", "idummy",
          "log_assets", "float_to_book", "float_to_earnings"]
CONTINUOUS = ["roe", "sales_growth_1y", "liquidity", "leverage", "log_assets", "float_to_book",
              "float_to_earnings"] + B_NEW

SPECS = {
    #             Outcome           beobachtet       Train        Test          Zusatz
    "main":      ("y_pe_main_h3",   "h3_observed",  (2012, 2017), (2020, 2022), {}),
    "pe_strict": ("y_pe_strict_h3", "h3_observed",  (2012, 2017), (2020, 2022), {}),
    "gp_broad":  ("y_gp_broad_h3",  "h3_observed",  (2012, 2017), (2020, 2022), {}),
    "h1":        ("y_pe_main_h1",   "h1_observed",  (2012, 2017), (2020, 2022), {}),
    "h5":        ("y_pe_main_h5",   "h5_observed",  (2012, 2015), (2020, 2020), {}),
    "complete":  ("y_pe_main_h3",   "h3_observed",  (2012, 2017), (2020, 2022), {"complete_case": True}),
    "assets10m": ("y_pe_main_h3",   "h3_observed",  (2012, 2017), (2020, 2022), {"min_assets": 10e6}),
    "noscreen":  ("y_pe_main_h3",   "h3_observed",  (2012, 2017), (2020, 2022), {"float_screen": False}),
    "nopartner": ("y_pe_main_h3",   "h3_observed",  (2012, 2017), (2020, 2022), {"no_partnerships": True}),
    "icov_split": ("y_pe_main_h3",  "h3_observed",  (2012, 2017), (2020, 2022), {"split_interest_coverage": True}),
}

def build_variant(key):
    """Stichprobe filtern, grdummy neu bauen, nur mit dem eigenen Training aufbereiten."""
    y_col, obs, (tr0, tr1), (te0, te1), opt = SPECS[key]
    s = full[full[obs]]
    s = s[~s["fin_sic_60_64"].astype(bool)]
    s = s[s["completion_status"] != "uncertain"]
    s = s[~pd.to_numeric(s["sic"], errors="coerce").isin([6770])]
    if opt.get("float_screen", True):
        s = s[(s["entity_public_float"] >= 50e6) | s["first_listing_year_exempt"].astype(bool)]
    if opt.get("no_partnerships"):
        s = s[~s["is_partnership"]]
    if "min_assets" in opt:
        s = s[~(s["total_assets"] < opt["min_assets"])]          # fehlende Bilanzsumme bleibt drin
    s = s.copy()
    s["y"] = s[y_col]

    # grdummy mit den Jahresmedianen DIESER Stichprobe (Regel aus Notebook 03)
    def high(col):
        return (s[col] > s.groupby("fiscal_year")[col].transform("median")).where(s[col].notna())
    g, l, v = high("sales_growth_1y"), high("liquidity"), high("leverage")
    combo = ((g == False) & (l == True) & (v == False)) | ((g == True) & (l == False) & (v == True))
    s["grdummy"] = combo.astype(float).where(g.notna() & l.notna() & v.notna())

    s["split"] = np.where(s["fiscal_year"].between(tr0, tr1), "train",
                          np.where(s["fiscal_year"].between(te0, te1), "test", "embargo"))
    train = s["split"] == "train"

    if opt.get("complete_case"):
        s = s.dropna(subset=CONTINUOUS + ["grdummy", "idummy"]).copy()   # ohne Imputation, ohne Indikatoren
        train = s["split"] == "train"
        indicators = []
    else:
        indicators = []
        for c in PALEPU + B_NEW:                                  # Indikator-Regel nur aus dem Training
            miss = s[c].isna()
            if miss[train].mean() > 0.02 and s.loc[train & miss, "y"].sum() >= 5:
                s[f"{c}_missing"] = miss.astype(int)
                indicators.append(f"{c}_missing")
        palepu_ind = [i for i in indicators if i.replace("_missing", "") in PALEPU]
        new_ind = [i for i in indicators if i not in palepu_ind]
        for i in new_ind:                                         # Entscheidung 29.10.: |rho| > 0.7 zu Palepu-Indikator
            if palepu_ind and s.loc[train, palepu_ind].corrwith(s.loc[train, i], method="spearman").abs().max() > 0.7:
                indicators.remove(i)
                s = s.drop(columns=i)
        if opt.get("split_interest_coverage"):
            # Check 9: interest_coverage_missing in zwei Indikatoren aufteilen (Definitionen aus src/15_compute_ratios.py).
            # interest_coverage fehlt, wenn der Zinsaufwand nicht > 0 ist ODER das operative Ergebnis fehlt.
            miss = s["interest_coverage"].isna()                     # Rohwert, vor Kappen und Auffüllen
            no_int = s["no_interest_expense"] == 1                   # Flag aus src/15: kein Zinsaufwand (0 oder ohne Schulden)
            assert (miss | ~no_int).all()                            # kein Zinsaufwand -> Deckung immer leer
            s["no_interest_expense"] = (miss & no_int).astype(int)
            s["coverage_missing_other"] = (miss & ~no_int).astype(int)
            pos = indicators.index("interest_coverage_missing")
            indicators[pos:pos + 1] = ["no_interest_expense", "coverage_missing_other"]
            s = s.drop(columns="interest_coverage_missing")

    lo, hi = s.loc[train, CONTINUOUS].quantile(0.01), s.loc[train, CONTINUOUS].quantile(0.99)
    s[CONTINUOUS] = s[CONTINUOUS].clip(lo, hi, axis=1)
    if not opt.get("complete_case"):
        med = s.loc[train, CONTINUOUS + ["idummy"]].median()
        s[CONTINUOUS + ["idummy"]] = s[CONTINUOUS + ["idummy"]].fillna(med)
        s["grdummy"] = s["grdummy"].fillna(0)

    s["log_assets_c"] = s["log_assets"] - s.loc[train, "log_assets"].mean()
    s["log_assets_c2"] = s["log_assets_c"] ** 2
    b_cols = PALEPU + B_NEW + indicators
    b2_cols = [("log_assets_c" if c == "log_assets" else c) for c in b_cols]
    b2_cols.insert(b2_cols.index("log_assets_c") + 1, "log_assets_c2")
    return s, b_cols, b2_cols, indicators

In [3]:
def fit(d, cols):
    return sm.Logit(d["y"], sm.add_constant(d[cols])).fit(
        method="newton", maxiter=200, disp=False, cov_type="cluster", cov_kwds={"groups": d["cik"]})

def boot_test(y, p, firms, seed=20261029, n=1000):
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(n):
        idx = np.concatenate([firms[i] for i in rng.integers(0, len(firms), len(firms))])
        if y[idx].sum() == 0 or y[idx].sum() == len(idx):
            continue
        out.append((roc_auc_score(y[idx], p[idx]), average_precision_score(y[idx], p[idx])))
    return np.array(out)

CORE = ["fcf_to_assets", "interest_coverage", "interest_coverage_missing", "altman_z2", "grdummy", "idummy",
        "leverage", "log_assets_c", "log_assets_c2"]
results = {}

def run_check(key):
    s, b_cols, b2_cols, ind = build_variant(key)
    tr, te = s[s["split"] == "train"], s[s["split"] == "test"].reset_index(drop=True)
    res_b2, res_b = fit(tr, b2_cols), fit(tr, b_cols)
    p_te = res_b.predict(sm.add_constant(te[b_cols], has_constant="add")).to_numpy()
    y_te = te["y"].to_numpy()
    draws = boot_test(y_te, p_te, list(te.groupby("cik").indices.values()))
    results[key] = {"res_b2": res_b2, "res_b": res_b, "indicators": ind,
                    "counts": {part: (len(d), int(d["y"].sum()), d.loc[d["y"] == 1, "cik"].nunique())
                               for part, d in [("Train", tr), ("Test", te)]},
                    "years": (tr["fiscal_year"].min(), tr["fiscal_year"].max(), te["fiscal_year"].min(), te["fiscal_year"].max()),
                    "auc": (roc_auc_score(y_te, p_te), *np.quantile(draws[:, 0], [.025, .975])),
                    "pr": (average_precision_score(y_te, p_te), *np.quantile(draws[:, 1], [.025, .975])),
                    "base_rate": y_te.mean(), "converged": res_b2.mle_retvals["converged"] and res_b.mle_retvals["converged"]}
    return results[key]

def report(key):
    r, ref = results[key], results["main"]["res_b2"]
    (n_tr, p_tr, f_tr), (n_te, p_te, f_te) = r["counts"]["Train"], r["counts"]["Test"]
    y0, y1, t0, t1 = r["years"]
    print(f"Train FY{y0}-{y1}: {n_tr:,} Zeilen, {p_tr} Positive ({f_tr} Firmen) | "
          f"Test FY{t0}-{t1}: {n_te:,} Zeilen, {p_te} Positive ({f_te} Firmen) | konvergiert: {r['converged']}")
    print(f"Indikatoren: {r['indicators'] or 'keine'}")
    rows = []
    for v in CORE:
        if v not in r["res_b2"].params.index:
            rows.append({"Variable": v, "Odds Ratio": np.nan, "p-Wert": np.nan, "Richtung wie im Hauptmodell?": "nicht im Modell"})
            continue
        c, p = r["res_b2"].params[v], r["res_b2"].pvalues[v]
        same = np.sign(c) == np.sign(ref.params[v]) if v in ref.params.index else None
        rows.append({"Variable": v, "Odds Ratio": np.exp(c), "p-Wert": p,
                     "Richtung wie im Hauptmodell?": "ja" if same else "nein"})
    display(pd.DataFrame(rows).set_index("Variable").round(4))
    a, pr = r["auc"], r["pr"]
    print(f"Test, Modell B: ROC-AUC {a[0]:.3f} [{a[1]:.3f}, {a[2]:.3f}] | PR-AUC {pr[0]:.4f} [{pr[1]:.4f}, {pr[2]:.4f}] "
          f"| Basisrate {r['base_rate']:.2%}")

## B. Referenz: Hauptspezifikation (muss Notebook 06 reproduzieren)

Kontrolle der Funktion: B2 muss McFadden-R² 5,43 % und `log_assets²`-Koeffizient −0,0807 ergeben, Modell B im Test ROC-AUC 0,698 [0,622; 0,769].

In [4]:
run_check("main")
print(f"Kontrolle: B2 McFadden-R² {results['main']['res_b2'].prsquared:.2%} | log_assets_c2 {results['main']['res_b2'].params['log_assets_c2']:.4f}")
report("main")

Kontrolle: B2 McFadden-R² 5.43% | log_assets_c2 -0.0807
Train FY2012-2017: 18,058 Zeilen, 167 Positive (76 Firmen) | Test FY2020-2022: 9,596 Zeilen, 70 Positive (43 Firmen) | konvergiert: True
Indikatoren: ['roe_missing', 'sales_growth_1y_missing', 'liquidity_missing', 'grdummy_missing', 'float_to_book_missing', 'float_to_earnings_missing', 'fcf_to_assets_missing', 'interest_coverage_missing', 'altman_z2_missing']


,Odds Ratio,p-Wert,Richtung wie im Hauptmodell?
Variable,,,
fcf_to_assets,3.4220,0.1484,ja
interest_coverage,0.9996,0.0024,ja
interest_coverage_missing,0.4314,0.0234,ja
altman_z2,0.9886,0.0537,ja
grdummy,0.4577,0.0037,ja
idummy,1.3108,0.1566,ja
leverage,0.6504,0.3396,ja
log_assets_c,1.0241,0.7883,ja
log_assets_c2,0.9225,0.0104,ja


Test, Modell B: ROC-AUC 0.698 [0.622, 0.769] | PR-AUC 0.0166 [0.0101, 0.0348] | Basisrate 0.73%


## 1. Outcome pe_strict

In [5]:
run_check("pe_strict")
report("pe_strict")

Train FY2012-2017: 18,058 Zeilen, 146 Positive (65 Firmen) | Test FY2020-2022: 9,596 Zeilen, 66 Positive (40 Firmen) | konvergiert: True
Indikatoren: ['roe_missing', 'sales_growth_1y_missing', 'liquidity_missing', 'grdummy_missing', 'float_to_book_missing', 'float_to_earnings_missing', 'fcf_to_assets_missing', 'interest_coverage_missing', 'altman_z2_missing']


,Odds Ratio,p-Wert,Richtung wie im Hauptmodell?
Variable,,,
fcf_to_assets,2.7824,0.2765,ja
interest_coverage,0.9996,0.0053,ja
interest_coverage_missing,0.3833,0.0201,ja
altman_z2,0.9874,0.0561,ja
grdummy,0.3882,0.0024,ja
idummy,1.4401,0.0640,ja
leverage,0.4952,0.1723,ja
log_assets_c,1.0954,0.3300,ja
log_assets_c2,0.9166,0.0143,ja


Test, Modell B: ROC-AUC 0.722 [0.646, 0.794] | PR-AUC 0.0194 [0.0107, 0.0515] | Basisrate 0.69%


**Ergebnis:** Mit der strengen Definition (nur institutionelle PE-Fonds; 146 statt 167 Trainings-Ziele) halten alle Kernbefunde: Zinsdeckung, fehlender Zinsaufwand, Altman-Z (10 %), `grdummy` und Größe² haben dieselbe Richtung und bleiben signifikant. `idummy` wird sogar signifikant positiv (p = 0,06). Die Test-AUC ist mit 0,72 etwas höher als im Hauptmodell – das Profil passt zu „echten“ PE-Fonds eher noch besser.

## 2. Outcome gp_broad (alle Going-Privates)

In [6]:
run_check("gp_broad")
report("gp_broad")

Train FY2012-2017: 18,058 Zeilen, 1873 Positive (838 Firmen) | Test FY2020-2022: 9,596 Zeilen, 877 Positive (536 Firmen) | konvergiert: True
Indikatoren: ['roe_missing', 'sales_growth_1y_missing', 'liquidity_missing', 'leverage_missing', 'grdummy_missing', 'float_to_book_missing', 'float_to_earnings_missing', 'fcf_to_assets_missing', 'interest_coverage_missing', 'altman_z2_missing']


,Odds Ratio,p-Wert,Richtung wie im Hauptmodell?
Variable,,,
fcf_to_assets,1.1357,0.7972,ja
interest_coverage,0.9999,0.2360,ja
interest_coverage_missing,0.8766,0.1406,ja
altman_z2,0.9922,0.0030,ja
grdummy,0.9650,0.6707,ja
idummy,1.1792,0.0064,ja
leverage,1.3491,0.0831,nein
log_assets_c,1.0861,0.0073,ja
log_assets_c2,0.9379,0.0000,ja


Test, Modell B: ROC-AUC 0.559 [0.531, 0.584] | PR-AUC 0.1165 [0.1001, 0.1385] | Basisrate 9.14%


**Ergebnis:** Für **alle** Going-Privates (Basisrate 9 %, rund zehnmal so viele Ziele) bricht das Profil weitgehend zusammen: Zinsdeckung, fehlender Zinsaufwand und `grdummy` sind nicht mehr signifikant, Verschuldung kehrt sich sogar um (positiv, p = 0,08). Es halten nur Altman-Z (negativ), `idummy` (positiv) und das umgekehrte U der Größe. Die Test-AUC fällt auf **0,56**. Das ist ein wichtiger Befund: Das gefundene Profil ist **PE-spezifisch** – andere Going-Privates (Familie, Management, strategische Käufer, Distress-Fälle) sehen anders aus.

## 3a. Horizont h = 1

In [7]:
run_check("h1")
report("h1")

Train FY2012-2017: 18,058 Zeilen, 51 Positive (51 Firmen) | Test FY2020-2022: 9,596 Zeilen, 19 Positive (19 Firmen) | konvergiert: True
Indikatoren: ['roe_missing', 'sales_growth_1y_missing', 'liquidity_missing', 'grdummy_missing', 'fcf_to_assets_missing', 'interest_coverage_missing', 'altman_z2_missing']


,Odds Ratio,p-Wert,Richtung wie im Hauptmodell?
Variable,,,
fcf_to_assets,49.8451,0.0012,ja
interest_coverage,0.9996,0.0000,ja
interest_coverage_missing,0.3509,0.0246,ja
altman_z2,0.9838,0.0728,ja
grdummy,0.4518,0.1063,ja
idummy,0.9526,0.8835,nein
leverage,0.6647,0.4389,ja
log_assets_c,0.9835,0.8647,nein
log_assets_c2,0.9320,0.0786,ja


Test, Modell B: ROC-AUC 0.663 [0.537, 0.786] | PR-AUC 0.0046 [0.0020, 0.0189] | Basisrate 0.20%


**Ergebnis:** Mit h = 1 (Ziel im **nächsten** Jahr) gibt es nur 51 Ziele im Training und 19 im Test – sehr wenig. Zinsdeckung, fehlender Zinsaufwand, Altman-Z und Größe² halten; der freie Cashflow ist hier sogar stark signifikant (p = 0,001). `grdummy` bleibt negativ, verliert aber knapp die Signifikanz (p = 0,11). Das Test-Intervall (0,54–0,79) ist wegen der 19 Ziele sehr breit.

## 3b. Horizont h = 5

In [8]:
run_check("h5")
report("h5")

Train FY2012-2015: 12,300 Zeilen, 165 Positive (71 Firmen) | Test FY2020-2020: 2,935 Zeilen, 32 Positive (32 Firmen) | konvergiert: True
Indikatoren: ['roe_missing', 'sales_growth_1y_missing', 'liquidity_missing', 'grdummy_missing', 'float_to_book_missing', 'float_to_earnings_missing', 'fcf_to_assets_missing', 'interest_coverage_missing', 'cash_ratio_missing', 'altman_z2_missing']


,Odds Ratio,p-Wert,Richtung wie im Hauptmodell?
Variable,,,
fcf_to_assets,3.5296,0.2945,ja
interest_coverage,0.9996,0.0059,ja
interest_coverage_missing,0.4772,0.0840,ja
altman_z2,0.9889,0.1926,ja
grdummy,0.3940,0.0028,ja
idummy,1.0737,0.7256,ja
leverage,0.5757,0.2345,ja
log_assets_c,0.9591,0.6646,nein
log_assets_c2,0.9181,0.0100,ja


Test, Modell B: ROC-AUC 0.654 [0.567, 0.729] | PR-AUC 0.0229 [0.0124, 0.0626] | Basisrate 1.09%


**Ergebnis:** Mit h = 5 (Training FY2012–2015, Test nur FY2020) halten Zinsdeckung, fehlender Zinsaufwand (10 %), `grdummy` und Größe²; Altman-Z bleibt negativ, ist aber nicht mehr signifikant. Die Test-AUC von 0,65 beruht auf nur einem Testjahr mit 32 Zielen.

## 4. Complete-Case

In [9]:
run_check("complete")
report("complete")

Train FY2013-2017: 6,827 Zeilen, 74 Positive (39 Firmen) | Test FY2020-2022: 4,282 Zeilen, 45 Positive (28 Firmen) | konvergiert: True
Indikatoren: keine


,Odds Ratio,p-Wert,Richtung wie im Hauptmodell?
Variable,,,
fcf_to_assets,11.2548,0.1359,ja
interest_coverage,0.9997,0.0004,ja
interest_coverage_missing,NaN,NaN,nicht im Modell
altman_z2,0.9705,0.2020,ja
grdummy,0.3035,0.0017,ja
idummy,1.5786,0.0643,ja
leverage,0.2185,0.1372,ja
log_assets_c,0.9968,0.9778,nein
log_assets_c2,0.9519,0.3080,ja


Test, Modell B: ROC-AUC 0.614 [0.508, 0.721] | PR-AUC 0.0187 [0.0097, 0.0464] | Basisrate 1.05%


**Ergebnis:** Nur vollständige Zeilen zu nehmen kostet mehr als die Hälfte der Ziele (74 statt 167 im Training; FY2012 fällt ganz weg, weil das Umsatzwachstum dort fehlt). **Wichtig:** Damit fallen auch **alle Firmen ohne Zinsaufwand** heraus (für sie ist die Zinsdeckung nicht definiert) – genau die Gruppe, die im Hauptmodell deutlich seltener Ziel ist; der Indikator kann deshalb gar nicht geschätzt werden. Zinsdeckung und `grdummy` halten trotzdem, `idummy` wird signifikant positiv (10 %), Altman-Z und Größe² behalten die Richtung, verlieren aber die Signifikanz. Test-AUC 0,61 mit sehr breitem Intervall.

## 5. Ohne Bilanzsumme < 10 Mio. USD

In [10]:
run_check("assets10m")
report("assets10m")

Train FY2012-2017: 17,139 Zeilen, 167 Positive (76 Firmen) | Test FY2020-2022: 9,327 Zeilen, 70 Positive (43 Firmen) | konvergiert: True
Indikatoren: ['roe_missing', 'sales_growth_1y_missing', 'liquidity_missing', 'grdummy_missing', 'float_to_book_missing', 'float_to_earnings_missing', 'fcf_to_assets_missing', 'interest_coverage_missing', 'altman_z2_missing']


,Odds Ratio,p-Wert,Richtung wie im Hauptmodell?
Variable,,,
fcf_to_assets,3.3031,0.1928,ja
interest_coverage,0.9995,0.0017,ja
interest_coverage_missing,0.4376,0.0268,ja
altman_z2,0.9892,0.2797,ja
grdummy,0.4562,0.0034,ja
idummy,1.3019,0.1652,ja
leverage,0.7329,0.5631,ja
log_assets_c,0.9456,0.5417,nein
log_assets_c2,0.9283,0.0380,ja


Test, Modell B: ROC-AUC 0.706 [0.633, 0.771] | PR-AUC 0.0170 [0.0099, 0.0333] | Basisrate 0.75%


**Ergebnis:** Ohne Firmen mit weniger als 10 Mio. USD Bilanzsumme fallen rund 900 Trainingszeilen weg, aber **kein einziges Ziel**. Alle Kernbefunde halten außer Altman-Z (gleiche Richtung, p = 0,28). Insbesondere bleibt das **umgekehrte U der Größe** signifikant (p = 0,04): Es entsteht also nicht nur durch die Kleinstfirmen. Test-AUC 0,71, praktisch wie im Hauptmodell.

## 6. Ohne $50M-Float-Filter

In [11]:
run_check("noscreen")
report("noscreen")

Train FY2012-2017: 29,006 Zeilen, 184 Positive (85 Firmen) | Test FY2020-2022: 13,352 Zeilen, 76 Positive (45 Firmen) | konvergiert: True
Indikatoren: ['roe_missing', 'sales_growth_1y_missing', 'liquidity_missing', 'grdummy_missing', 'float_to_book_missing', 'float_to_earnings_missing', 'fcf_to_assets_missing', 'fcf_vol_missing', 'interest_coverage_missing', 'cash_ratio_missing', 'altman_z2_missing']


,Odds Ratio,p-Wert,Richtung wie im Hauptmodell?
Variable,,,
fcf_to_assets,1.7758,0.4402,ja
interest_coverage,0.9997,0.0076,ja
interest_coverage_missing,0.5420,0.0647,ja
altman_z2,0.9982,0.0249,ja
grdummy,0.4852,0.0100,ja
idummy,1.2876,0.1658,ja
leverage,0.6850,0.3602,ja
log_assets_c,1.8618,0.0000,ja
log_assets_c2,0.8854,0.0001,ja


Test, Modell B: ROC-AUC 0.744 [0.678, 0.805] | PR-AUC 0.0174 [0.0098, 0.0370] | Basisrate 0.57%


**Ergebnis:** Ohne den $50M-Float-Filter wächst das Training auf 29.006 Zeilen (184 Ziele). Alle Kernbefunde halten (Zinsdeckung, fehlender Zinsaufwand, Altman-Z, `grdummy`, Größe²). Der lineare Größenterm wird stark positiv, weil jetzt sehr viele Kleinstfirmen ohne Ziele dabei sind. Die Test-AUC ist mit 0,74 höher – vor allem, weil Kleinstfirmen leicht als Nicht-Ziele zu erkennen sind, nicht weil das Modell PE-Ziele besser findet.

## Vorbereitung Check 7: Partnership-Kennzeichen

Partnerships (LPs/MLPs) melden ihr Eigenkapital als „partners' capital“, das die Eigenkapital-Regel der Pipeline nicht erfasst; deshalb fehlen bei ihnen `roe` und `float_to_book` systematisch (Notebook 09). **Entscheidung 30.10.2026:** keine Pipeline-Änderung, sondern ein Kennzeichen `is_partnership` über den Firmennamen.

**Muster** (Groß-/Kleinschreibung egal): „LP“, „L.P.“ oder „L P“ als eigenes Wort (danach Namensende, Leerzeichen, Komma, Klammer oder „/“) · „LIMITED PARTNERSHIP“ · „PARTNERSHIP“ am Namensende · „MLP“ als eigenes Wort. „PARTNERSHIP“ mitten im Namen zählt nicht (sonst würde z. B. der SPAC „Atlantic Alliance Partnership Corp.“ markiert). Das Kennzeichen gilt je Firmenjahr, also für den Namen im jeweiligen 10-K (eine Firma kann ihre Rechtsform ändern).

In [12]:
names = full.drop_duplicates("company_name")["company_name"]
flagged = names[names.str.contains(PARTNERSHIP_PATTERN)]
partners_not_flagged = names[names.str.contains("PARTNERS", case=False) & ~names.str.contains(PARTNERSHIP_PATTERN)]
print(f"Verschiedene Firmennamen im Panel: {len(names):,} | markiert: {len(flagged)} | "
      f"enthalten 'Partners', aber nicht markiert: {len(partners_not_flagged)}")
pd.DataFrame({"20 zufällige markierte Namen": flagged.sample(20, random_state=1).to_numpy(),
              "20 Namen mit 'Partners', NICHT markiert": partners_not_flagged.sample(20, random_state=1).to_numpy()})

Verschiedene Firmennamen im Panel: 14,068 | markiert: 252 | enthalten 'Partners', aber nicht markiert: 65


,20 zufällige markierte Namen,"20 Namen mit 'Partners', NICHT markiert"
0,TIDEWATER FUTURES FUND LP,INTERPRIVATE III FINANCIAL PARTNERS INC.
1,MACH NATURAL RESOURCES LP,"GORES TECHNOLOGY PARTNERS II, INC."
2,LANDMARK INFRASTRUCTURE PARTNERS LP,HIGHLAND TRANSCEND PARTNERS I CORP.
3,MANAGED FUTURES PREMIER BHM L.P.,PARTNERS BANCORP
4,MEWBOURNE ENERGY PARTNERS 04-A LP,"ADVANCED MERGER PARTNERS, INC."
5,"DOMINION ENERGY MIDSTREAM PARTNERS, LP",ARCHIMEDES TECH SPAC PARTNERS CO
6,CERES TACTICAL CURRENCY L.P.,"DIGITAL DEVELOPMENT PARTNERS, INC."
7,HESS MIDSTREAM LP,NISKA GAS STORAGE PARTNERS LLC
8,LEHIGH GAS PARTNERS LP,"GLOBAL SPAC PARTNERS CO,"
9,"ROYAL HAWAIIAN ORCHARDS, L.P.",ACAMAR PARTNERS ACQUISITION CORP.


In [13]:
# Grenzfälle zur Prüfung: "Partners LLC" (oft MLP-ähnlich aufgebaut) und "Partnership" mitten im Namen
borderline = names[names.str.contains(r"PARTNERS,? LLC|PARTNERSHIP\b(?!\W*$)", case=False, regex=True)
                   & ~names.str.contains(PARTNERSHIP_PATTERN)]
print("Grenzfälle (nicht markiert):", borderline.tolist())

Grenzfälle (nicht markiert): ['GOLDMAN SACHS HEDGE FUND PARTNERS LLC', 'CONSTELLATION ENERGY PARTNERS LLC', 'SANCHEZ PRODUCTION PARTNERS LLC', 'NISKA GAS STORAGE PARTNERS LLC', 'TRANSOCEAN PARTNERS LLC', 'ATLANTIC ALLIANCE PARTNERSHIP CORP.']


In [14]:
# Wie viele Partnerships und Ziele sind in der Hauptstichprobe?
s_main, *_ = build_variant("main")
rows = []
for part in ["train", "embargo", "test"]:
    d = s_main[s_main["split"] == part]
    pt = d[d["is_partnership"]]
    rows.append({"Abschnitt": part, "Zeilen": len(d), "Partnership-Zeilen": len(pt),
                 "Partnership-Firmen": pt["cik"].nunique(), "Ziele (Zeilen)": int(d["y"].sum()),
                 "Ziele bei Partnerships (Zeilen)": int(pt["y"].sum()),
                 "Ziel-Firmen Partnerships": pt.loc[pt["y"] == 1, "cik"].nunique()})
display(pd.DataFrame(rows).set_index("Abschnitt"))
print(f"Partnership-Firmen in der ganzen Stichprobe: {s_main.loc[s_main['is_partnership'], 'cik'].nunique()} von {s_main['cik'].nunique():,}")
print(f"Ziel-Anteil: Partnerships {s_main.loc[s_main['is_partnership'], 'y'].mean():.2%} | übrige {s_main.loc[~s_main['is_partnership'], 'y'].mean():.2%}")

,Zeilen,Partnership-Zeilen,Partnership-Firmen,Ziele (Zeilen),Ziele bei Partnerships (Zeilen),Ziel-Firmen Partnerships
Abschnitt,,,,,,
train,18058,633,151,167,14,6
embargo,5774,140,80,45,6,4
test,9596,149,59,70,3,2


Partnership-Firmen in der ganzen Stichprobe: 153 von 6,466
Ziel-Anteil: Partnerships 2.49% | übrige 0.80%


In [15]:
# Zusammenhang mit den Eigenkapital-Indikatoren (Hauptstichprobe, alle Abschnitte)
share_all = s_main["is_partnership"].mean()
rows = []
for ind in ["float_to_book_missing", "roe_missing"]:
    miss = s_main[ind] == 1
    rows.append({"Indikator": ind, "Zeilen mit fehlendem Wert": int(miss.sum()),
                 "Anteil Partnerships unter den fehlenden": f"{s_main.loc[miss, 'is_partnership'].mean():.1%}",
                 "Anteil fehlend bei Partnerships": f"{s_main.loc[s_main['is_partnership'], ind].mean():.1%}",
                 "Anteil fehlend bei übrigen": f"{s_main.loc[~s_main['is_partnership'], ind].mean():.1%}",
                 "Spearman rho mit is_partnership": round(s_main[ind].corr(s_main["is_partnership"].astype(int), method="spearman"), 3)})
print(f"Anteil Partnerships an allen Zeilen: {share_all:.1%}")
pd.DataFrame(rows).set_index("Indikator")

Anteil Partnerships an allen Zeilen: 2.8%


,Zeilen mit fehlendem Wert,Anteil Partnerships unter den fehlenden,Anteil fehlend bei Partnerships,Anteil fehlend bei übrigen,Spearman rho mit is_partnership
Indikator,,,,,
float_to_book_missing,2436,33.0%,87.3%,5.0%,0.518
roe_missing,3946,20.4%,87.2%,9.7%,0.394


**Ergebnis der Vorbereitung:**

- **Regel:** Von 14.068 verschiedenen Firmennamen werden 252 markiert. Die Stichprobe zeigt typische MLPs (z. B. MPLX, Buckeye, Cheniere Energy Partners), aber auch Rohstoff- und Futures-Fonds in LP-Form. Von den 65 Namen mit „Partners“, die **nicht** markiert werden, sind fast alle Kapitalgesellschaften (Inc./Corp.) oder SPACs – richtig so.
- **Grenzfälle zur Entscheidung:** fünf LLCs mit „Partners“ im Namen, die MLP-ähnlich aufgebaut sein können (z. B. Constellation Energy Partners LLC, Transocean Partners LLC, Niska Gas Storage Partners LLC); sie sind nach der Regel **nicht** markiert.
- **Umfang:** In der Hauptstichprobe sind **153 von 6.466 Firmen** Partnerships (2,4 %). Unter den Zielen sind es im Training 14 Zeilen von 6 Firmen, im Embargo 6 von 4, im Test 3 von 2. Partnerships werden aber **dreimal so oft Ziel** wie andere Firmen (2,5 % gegenüber 0,8 % der Zeilen) – die Rechtsform hängt also tatsächlich mit Take-Privates zusammen, es ist nicht nur eine Datenlücke.
- **Zusammenhang mit den Indikatoren:** Bei Partnerships fehlen `float_to_book` und `roe` in **87 %** der Zeilen, bei anderen Firmen nur in 5 % bzw. 10 %. Obwohl Partnerships nur 2,8 % der Zeilen stellen, machen sie **33 %** aller fehlenden `float_to_book`-Werte aus (Spearman rho 0,52) und 20 % der fehlenden `roe`-Werte (rho 0,39). `float_to_book_missing` misst damit zu einem großen Teil „ist eine Partnership“.

## 7. Ohne Partnerships

In [16]:
run_check("nopartner")
report("nopartner")

Train FY2012-2017: 17,425 Zeilen, 153 Positive (70 Firmen) | Test FY2020-2022: 9,447 Zeilen, 67 Positive (41 Firmen) | konvergiert: True
Indikatoren: ['roe_missing', 'sales_growth_1y_missing', 'liquidity_missing', 'grdummy_missing', 'float_to_book_missing', 'fcf_to_assets_missing', 'interest_coverage_missing']


,Odds Ratio,p-Wert,Richtung wie im Hauptmodell?
Variable,,,
fcf_to_assets,4.2885,0.1025,ja
interest_coverage,0.9996,0.0025,ja
interest_coverage_missing,0.3932,0.0158,ja
altman_z2,0.9894,0.0619,ja
grdummy,0.5240,0.0082,ja
idummy,1.3531,0.1213,ja
leverage,0.7362,0.5255,ja
log_assets_c,1.0319,0.7389,ja
log_assets_c2,0.9275,0.0142,ja


Test, Modell B: ROC-AUC 0.685 [0.608, 0.746] | PR-AUC 0.0130 [0.0081, 0.0196] | Basisrate 0.71%


**Ergebnis:** Ohne Partnerships (153 statt 167 Trainings-Ziele, 67 statt 70 im Test) halten **alle Kernbefunde**: Zinsdeckung (p = 0,003), fehlender Zinsaufwand (p = 0,02), Altman-Z (p = 0,06), `grdummy` (p = 0,008) und das umgekehrte U der Größe (p = 0,01) – alle mit derselben Richtung wie im Hauptmodell. Die Test-AUC bleibt mit 0,685 praktisch gleich (Hauptmodell 0,698). Die Partnerships verzerren also die Watchlist, aber nicht die Kernbefunde des Modells.

## Übersicht: Welche Kernbefunde halten?

Zeichen je Variable: **✓** = gleiche Richtung wie im Hauptmodell **und** p < 0,10; **○** = gleiche Richtung, aber p ≥ 0,10; **✗** = umgekehrte Richtung. In der Zeile der Hauptspezifikation zeigt ✓/○, ob der Befund dort selbst signifikant ist. „Kernbefunde“ sind die Variablen, die im Hauptmodell p < 0,10 haben.

In [17]:
CHECKS = [('main', 'Hauptspezifikation (Referenz)'), ('pe_strict', '1. Outcome pe_strict'), ('gp_broad', '2. Outcome gp_broad (alle Going-Privates)'), ('h1', '3a. Horizont h = 1'), ('h5', '3b. Horizont h = 5'), ('complete', '4. Complete-Case'), ('assets10m', '5. Ohne Bilanzsumme < 10 Mio. USD'), ('noscreen', '6. Ohne $50M-Float-Filter'), ('nopartner', '7. Ohne Partnerships')]
ref = results["main"]["res_b2"]
labels = dict(CHECKS)
rows = []
for key, title in CHECKS:
    r = results[key]
    row = {"Check": title}
    for v in CORE:
        if v not in r["res_b2"].params.index:
            row[v] = "–"
            continue
        c, p = r["res_b2"].params[v], r["res_b2"].pvalues[v]
        if np.sign(c) != np.sign(ref.params[v]):
            row[v] = "✗"
        else:
            row[v] = "✓" if p < 0.10 else "○"
    a = r["auc"]
    row["Test ROC-AUC (B)"] = f"{a[0]:.3f} [{a[1]:.2f}; {a[2]:.2f}]"
    row["Positive Train/Test"] = f"{r['counts']['Train'][1]}/{r['counts']['Test'][1]}"
    rows.append(row)
overview = pd.DataFrame(rows).set_index("Check")
core_findings = [v for v in CORE if ref.pvalues[v] < 0.10]
print("Kernbefunde im Hauptmodell (p < 0.10):", core_findings)
held = {v: sum(overview.loc[overview.index != labels["main"], v] == "✓") for v in core_findings}
print(f"Wie oft gehalten (✓) in den {len(CHECKS) - 1} Checks:", held)
overview

Kernbefunde im Hauptmodell (p < 0.10): ['interest_coverage', 'interest_coverage_missing', 'altman_z2', 'grdummy', 'log_assets_c2']
Wie oft gehalten (✓) in den 8 Checks: {'interest_coverage': 7, 'interest_coverage_missing': 6, 'altman_z2': 5, 'grdummy': 6, 'log_assets_c2': 7}


,fcf_to_assets,interest_coverage,interest_coverage_missing,altman_z2,grdummy,idummy,leverage,log_assets_c,log_assets_c2,Test ROC-AUC (B),Positive Train/Test
Check,,,,,,,,,,,
Hauptspezifikation (Referenz),○,✓,✓,✓,✓,○,○,○,✓,0.698 [0.62; 0.77],167/70
1. Outcome pe_strict,○,✓,✓,✓,✓,✓,○,○,✓,0.722 [0.65; 0.79],146/66
2. Outcome gp_broad (alle Going-Privates),○,○,○,✓,○,✓,✗,✓,✓,0.559 [0.53; 0.58],1873/877
3a. Horizont h = 1,✓,✓,✓,✓,○,✗,○,✗,✓,0.663 [0.54; 0.79],51/19
3b. Horizont h = 5,○,✓,✓,○,✓,○,○,✗,✓,0.654 [0.57; 0.73],165/32
4. Complete-Case,○,✓,–,○,✓,✓,○,✗,○,0.614 [0.51; 0.72],74/45
5. Ohne Bilanzsumme < 10 Mio. USD,○,✓,✓,○,✓,○,○,✗,✓,0.706 [0.63; 0.77],167/70
6. Ohne $50M-Float-Filter,○,✓,✓,✓,✓,○,○,✓,✓,0.744 [0.68; 0.80],184/76
7. Ohne Partnerships,○,✓,✓,✓,✓,○,○,○,✓,0.685 [0.61; 0.75],153/67


**Ergebnis:** Kernbefunde im Hauptmodell (B2, p < 0,10) sind Zinsdeckung (−), fehlender Zinsaufwand (−), Altman-Z (−), `grdummy` (−) und Größe² (−, umgekehrtes U). Sie halten so oft (in 8 Checks):

- **Zinsdeckung:** 7 von 8 Checks (nur bei allen Going-Privates nicht).
- **Größe² (umgekehrtes U):** 7 von 8 (nur Complete-Case nicht signifikant, Richtung gleich).
- **Fehlender Zinsaufwand:** 6 von 8 (bei Complete-Case nicht schätzbar, bei allen Going-Privates nicht signifikant).
- **`grdummy`:** 6 von 8 (bei allen Going-Privates und h = 1 nicht signifikant, Richtung bei h = 1 gleich).
- **Altman-Z:** 5 von 8 – der schwächste Kernbefund; die Richtung ist aber überall gleich.

**Freier Cashflow** ist schon im Hauptmodell B2 nicht signifikant (p = 0,15) und hält nur bei h = 1. Sein signifikanter Effekt in Modell B (Notebook 06) verschwindet also, sobald die Größe nicht-linear berücksichtigt wird – dieser Befund ist **nicht robust**. Ein „✗“ beim linearen Größenterm `log_assets_c` ist unproblematisch: Er beschreibt nur die Steigung genau am Mittelwert, die nahe null liegt. Ein echter Richtungswechsel tritt nur bei Verschuldung (alle Going-Privates) und `idummy` (h = 1, nicht signifikant) auf.

## Check 9: Indikator für fehlende Zinsdeckung aufgeteilt (Zusatz, 01.11.2026; in der Anhang-Tabelle Nr. 8, weil 3a/3b dort eine Nummer teilen)

`interest_coverage_missing` mischt drei Fälle (siehe Notebook 10): kein Zinsaufwand, Zinsaufwand trotz Schulden nicht getaggt, operatives Ergebnis fehlt. Hier wird er in **Modell B** durch zwei Indikatoren ersetzt, mit den Definitionen aus `src/15_compute_ratios.py`:

- **`no_interest_expense`** = Deckung fehlt, weil die Firma keinen Zinsaufwand hat (Zinsaufwand 0, oder nicht getaggt bei Schulden = 0).
- **`coverage_missing_other`** = Deckung fehlt aus einem anderen Grund (Zinsaufwand trotz Schulden nicht getaggt, oder operatives Ergebnis fehlt).

Sonst alles wie Modell B (gleiche Variablen, Aufbereitung aus dem Training FY2012–2017, ungewichtet, Cluster nach Firma, Test FY2020–2022). Wie bei den anderen Checks wird zusätzlich B2 geschätzt (für die Anhang-Tabelle). Dieser Check steht **nicht** in der Übersicht oben, damit die Auswertung der ersten 8 Checks unverändert bleibt.

In [18]:
run_check("icov_split")
r9 = results["icov_split"]
res9 = r9["res_b"]                                       # Modell B mit aufgeteiltem Indikator
show = ["no_interest_expense", "coverage_missing_other", "interest_coverage", "fcf_to_assets", "altman_z2", "grdummy"]
(n_tr, p_tr, f_tr), (n_te, p_te9, f_te) = r9["counts"]["Train"], r9["counts"]["Test"]
print(f"Modell B, Indikator aufgeteilt | Train {n_tr:,} Zeilen, {p_tr} Positive | McFadden-R² {res9.prsquared:.2%} "
      f"(Modell B: {results['main']['res_b'].prsquared:.2%}) | konvergiert: {res9.mle_retvals['converged']}")
print(f"Test: ROC-AUC {r9['auc'][0]:.3f} [{r9['auc'][1]:.3f}, {r9['auc'][2]:.3f}] (Modell B: {results['main']['auc'][0]:.3f}) | "
      f"PR-AUC {r9['pr'][0]:.4f}")
print("Indikatoren:", r9["indicators"])
pd.DataFrame({"Odds Ratio": np.exp(res9.params[show]), "p-Wert": res9.pvalues[show],
              "Modell B: Odds Ratio": np.exp(results["main"]["res_b"].params.reindex(show)),
              "Modell B: p-Wert": results["main"]["res_b"].pvalues.reindex(show)}).round(4)

Modell B, Indikator aufgeteilt | Train 18,058 Zeilen, 167 Positive | McFadden-R² 4.71% (Modell B: 4.55%) | konvergiert: True
Test: ROC-AUC 0.689 [0.610, 0.766] (Modell B: 0.698) | PR-AUC 0.0166
Indikatoren: ['roe_missing', 'sales_growth_1y_missing', 'liquidity_missing', 'grdummy_missing', 'float_to_book_missing', 'float_to_earnings_missing', 'fcf_to_assets_missing', 'no_interest_expense', 'coverage_missing_other', 'altman_z2_missing']


,Odds Ratio,p-Wert,Modell B: Odds Ratio,Modell B: p-Wert
no_interest_expense,0.2240,0.0202,NaN,NaN
coverage_missing_other,0.4999,0.0964,NaN,NaN
interest_coverage,0.9997,0.0099,0.9997,0.0090
fcf_to_assets,5.2473,0.0246,4.9544,0.0271
altman_z2,0.9889,0.0313,0.9881,0.0231
grdummy,0.4584,0.0040,0.4518,0.0031


In [19]:
# Wer sind die Firmenjahre mit no_interest_expense = 1? Schulden laut Pipeline (Drei-Zustands-Regel aus src/13c)
s9, *_ = build_variant("icov_split")
debt = pd.read_csv(DATA / "processed" / "firm_year_fundamentals.csv", usecols=["accession", "total_debt"])
s9 = s9.merge(debt, on="accession", how="left", validate="one_to_one")
tr9 = s9[s9["split"] == "train"]
ni = tr9[tr9["no_interest_expense"] == 1]
print(f"Training, no_interest_expense = 1: {len(ni):,} Firmenjahre")
print(f"  Gesamtschulden = 0: {(ni['total_debt'] == 0).mean():.1%} | Schulden fehlen: {ni['total_debt'].isna().mean():.1%} | "
      f"Schulden > 0: {(ni['total_debt'] > 0).mean():.1%}")
rows = []
for part in ["train", "test"]:
    d = s9[s9["split"] == part]
    for ind in ["no_interest_expense", "coverage_missing_other"]:
        g = d[d[ind] == 1]
        rows.append({"Abschnitt": part, "Gruppe": ind, "Firmenjahre": len(g), "Ziele (Zeilen)": int(g["y"].sum()),
                     "Ziel-Firmen": g.loc[g["y"] == 1, "cik"].nunique(), "Zielanteil": f"{g['y'].mean():.2%}"})
    rest = d[(d["no_interest_expense"] == 0) & (d["coverage_missing_other"] == 0)]
    rows.append({"Abschnitt": part, "Gruppe": "Deckung vorhanden", "Firmenjahre": len(rest), "Ziele (Zeilen)": int(rest["y"].sum()),
                 "Ziel-Firmen": rest.loc[rest["y"] == 1, "cik"].nunique(), "Zielanteil": f"{rest['y'].mean():.2%}"})
pd.DataFrame(rows).set_index(["Abschnitt", "Gruppe"])

Training, no_interest_expense = 1: 2,354 Firmenjahre
  Gesamtschulden = 0: 98.0% | Schulden fehlen: 0.1% | Schulden > 0: 1.9%


Firmenjahre  Ziele (Zeilen)  Ziel-Firmen  \
Abschnitt Gruppe                                                             
train     no_interest_expense            2354               6            4   
          coverage_missing_other         3363              20           10   
          Deckung vorhanden             12341             141           64   
test      no_interest_expense            1524               5            4   
          coverage_missing_other         1728               5            4   
          Deckung vorhanden              6344              60           36   

                                 Zielanteil  
Abschnitt Gruppe                             
train     no_interest_expense         0.25%  
          coverage_missing_other      0.59%  
          Deckung vorhanden           1.14%  
test      no_interest_expense         0.33%  
          coverage_missing_other      0.29%  
          Deckung vorhanden           0.95%

**Ergebnis Check 9:** Firmen **ohne Zinsaufwand** sind deutlich seltener Ziele (Odds Ratio 0,22, p = 0,02, gegenüber Firmen mit berechenbarer Zinsdeckung); 98 % von ihnen haben laut Pipeline überhaupt keine Schulden. Fehlt die Deckung aus anderen Gründen (Zinsdaten nicht getaggt oder operatives Ergebnis fehlt), ist der Effekt schwächer (OR 0,50, p = 0,10). Der gemeinsame Indikator in Modell B (OR 0,40) ist also vor allem ein „keine Schulden“-Effekt; die übrigen Befunde und die Test-AUC (0,689 statt 0,698) ändern sich kaum. Die Gruppen sind klein (im Training 6 bzw. 20 Ziel-Firmenjahre), die Intervalle entsprechend breit.

## Kernergebnisse

1. Die meisten Kernbefunde sind robust: Zinsdeckung, der Indikator „Zinsdeckung nicht berechenbar“, `grdummy` und das umgekehrte U der Größe halten in 6–7 von 8 Checks mit gleicher Richtung und p < 0,10.
2. Altman-Z hat überall dieselbe Richtung, ist aber nur in 5 von 8 Checks signifikant – ein schwächerer Befund.
3. Der freie Cashflow ist **nicht robust**: Schon im Hauptmodell mit Größe² (B2) ist er nicht signifikant; sein Effekt in Modell B hing an der linearen Größenvariable.
4. Das Profil ist **PE-spezifisch**: Für alle Going-Privates zusammen fallen die meisten Befunde weg, und die Test-AUC sinkt auf 0,56; mit der strengen PE-Definition ist es eher noch klarer (AUC 0,72).
5. Kleinstfirmen, Partnerships, der Float-Filter und andere Horizonte verändern die Kernaussagen nicht; die Vorhersagegüte im Test bleibt in allen PE-Varianten im Bereich 0,61–0,74, mit breiten Intervallen bei wenigen Zielen.